# Teil 3: Modell

In [ ]:
!pip install kagglehub pandas scikit-learn --quiet

import pandas as pd
import kagglehub
import os
import glob
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

#Nur erste 100000 Zeilen weil sonst zu viel
path = kagglehub.dataset_download("andrewmvd/steam-reviews")
csv_dateien = glob.glob(os.path.join(path, "*.csv"))
df = pd.read_csv(csv_dateien[0], nrows=100000)

print(f"Datensatz geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
df[['review_text', 'review_score']].head()

## 3.1 Aufteilung in Train- und Test-Satz
Der Datensatz wird 80/20 aufgeteilt: 80% für Training, 20% für Testing.
Leere Texte werden vorher entfernt, da sie keine Information liefern.

In [ ]:
# Zeilen mit leerem review_text oder fehlendem review_score entfernen
df_clean = df[['review_text', 'review_score']].dropna()
df_clean = df_clean[df_clean['review_text'].str.strip() != '']

X = df_clean['review_text']
y = df_clean['review_score']

# 80% Training, 20% Test; random_state das man es reproduzieren kann
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Trainingsdaten: {len(X_train)} Einträge")
print(f"Testdaten:      {len(X_test)} Einträge")
print(f"\nKlassenverteilung (Train):")
print(y_train.value_counts())

## 3.2 Algorithmus-Wahl und Modell-Training

**Gewählter Algorithmus: Logistic Regression mit TF-IDF Vektorisierung**

Da das Ziel-Feld `review_score` nur zwei Werte hat (`1` = positiv, `-1` = negativ), ist es ein binäres Klassifikationsproblem. Für Text-Klassifikation eigent sich dieser Ansatz besonders gut:

1. **TF-IDF Vectorizer**: Wandelt den Rohtext in numerische Merkmale um. TF-IDF gewichtet Wörter nach ihrer Häufigkeit im Dokument im Verhältnis zu ihrer Häufigkeit im gesamten Datensatz. Dadurch werden häufige, aber wenig aussagekräftige Wörter (wie "the", "and") automatisch weniger gewichtet.

2. **Logistic Regression**: Ist für Text-Klassifikation mit TF-IDF gut geeignet, weil sie schnell trainiert und gut interpretierbar ist. Verglichen mit komplexeren Modellen wie z. B. SVM liefert sie ähnlich gute Resultate aber braucht dafür viel weniger Rechenaufwand.

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f"Feature-Matrix (Train): {X_train_tfidf.shape}")

# Logistic Regression trainieren
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

## 3.3 Überprüfung des Modells
Ich generiere Vorhersagen auf den Test-Daten und überprüfe einige Beispiele manuell.

In [ ]:
# Vorhersagen auf Testdaten
y_pred = model.predict(X_test_tfidf)

# Genauigkeit
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print()
print("Klassifikationsbericht:")
print(classification_report(y_test, y_pred, target_names=['Negativ (-1)', 'Positiv (1)']))

In [ ]:
# Manuelle Überprüfung: 10 zufällige Beispiele aus dem Test-Set
import random
random.seed(7)

test_df = X_test.reset_index(drop=True).to_frame()
test_df['tatsächlich'] = y_test.reset_index(drop=True)
test_df['vorhergesagt'] = y_pred
test_df['korrekt'] = test_df['tatsächlich'] == test_df['vorhergesagt']

sample = test_df.sample(10, random_state=7)

label_map = {1: 'positiv', -1: 'negativ'}

for i, row in sample.iterrows():
    text_preview = str(row['review_text'])[:120].replace('\n', ' ')
    tats = label_map.get(row['tatsächlich'], row['tatsächlich'])
    vorh = label_map.get(row['vorhergesagt'], row['vorhergesagt'])
    korrekt = "korrekt" if row['korrekt'] else "falsch"
    print(f"Tatsächlich: {tats}, Vorhergesagt: {vorh} ({korrekt})")
    print(f"Text: {text_preview}")
    print()

In [ ]:
# eigene Texte testen
eigene_reviews = [
    "This game is absolutely amazing, I love it! Best purchase ever.",
    "Terrible game, full of bugs and not worth the money at all.",
    "It's okay, nothing special but not bad either.",
    "Do NOT buy this. Broken mechanics, horrible story, waste of time.",
    "One of the best games I've ever played. Highly recommend!"
]

eigene_vektoren = vectorizer.transform(eigene_reviews)
eigene_preds = model.predict(eigene_vektoren)

print("Vorhersagen für eigene Reviews:")
for text, pred in zip(eigene_reviews, eigene_preds):
    label = label_map.get(pred, pred)
    print(f"  [{label.upper():8s}] {text[:80]}")

## Fazit 3.3

Das Modell hat eine hohe Genauigkeit bei der Vorhersage von Steam-Reviews. Die manuell überprüften Beispiele zeigen, dass das Modell klar positive oder negative Texte richtig voneinander unterscheiden kann. Bei sehr kurzen oder mehrdeutigen Texten (z.B. "It's okay") ist die Vorhersage schwieriger, was an sich ja erwartet werden kann. Die eigenen Test-Reviews werden alle korrekt erkannt: deutlich positiver Text führt zur Klasse `positiv`, klar negativer zur Klasse `negativ`. Das Modell funktioniert gut für eindeutige Texte; Grenzfälle sind die grösste Schwäche.